# YOLOBERT — Kaggle GPU training

**Before running:** Settings → Accelerator = **GPU T4 x2** (or P100), Internet = **On**.

Sessions expire (~9-12h) and idle-timeout. Checkpoints go to `/kaggle/working` (persisted to output). Resume with `--resume`. Download `best.pth` before the session ends.

In [ ]:
# 1. GPU check
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 2. Clone repo. Public repo -> works as-is. Private -> use a token or Kaggle Secrets.
REPO = 'https://github.com/ShMazumder/YOLOBERT.git'
import os
if not os.path.isdir('/kaggle/working/YOLOBERT'):
    !git clone $REPO /kaggle/working/YOLOBERT
%cd /kaggle/working/YOLOBERT
!git pull -q || true

In [ ]:
# 3. Deps. torch/torchvision already on Kaggle image; install the rest.
!pip install -q pycocotools pyyaml tensorboard || pip install -q -r requirements.txt

In [ ]:
# 4. Smoke tests (fast, catch breakage before a long run)
!python tools/metrics.py
!python -c "import models; print('models:', list(models.MODELS))"
!pip install -q pytest && pytest tests/ -q

## Attach data
Right panel → **Add Data** → search **COCO 2017** → add. It mounts read-only under `/kaggle/input/<slug>/`. Set the paths below to match (check with `!ls /kaggle/input`).

In [ ]:
!ls /kaggle/input || echo 'no datasets attached yet'

In [ ]:
# 5. Write a Kaggle config (paths point at /kaggle/input + /kaggle/working)
import yaml, os
COCO = '/kaggle/input/coco-2017-dataset/coco2017'  # <-- adjust to your mount
cfg = {
    'seed': 42,
    'work_dir': '/kaggle/working/runs/fcos_r50',
    'model': 'fcos', 'model_backbone': 'resnet50', 'model_pretrained': True,
    'model_num_classes': 80,
    'dataset': 'coco', 'data_root': COCO,
    'train_img_dir': 'train2017', 'val_img_dir': 'val2017',
    'train_ann': f'{COCO}/annotations/instances_train2017.json',
    'val_ann':   f'{COCO}/annotations/instances_val2017.json',
    'img_size': 800, 'batch_size': 4, 'workers': 2,
    'lr': 0.01, 'weight_decay': 1e-4, 'epochs': 12,
    'metric': 'coco', 'metric_iou_type': 'bbox',
    'ann_file': f'{COCO}/annotations/instances_val2017.json',
    'primary_metric': 'AP',
}
os.makedirs('configs', exist_ok=True)
with open('configs/kaggle_coco.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)
print(open('configs/kaggle_coco.yaml').read())

In [ ]:
# 6. Train (auto-resume from last.pth if the session restarted)
import os
CKPT = '/kaggle/working/runs/fcos_r50/last.pth'
resume = f'--resume {CKPT}' if os.path.isfile(CKPT) else ''
!python tools/train.py --config configs/kaggle_coco.yaml $resume

In [ ]:
# 7. Evaluate best checkpoint -> COCO AP
!python tools/test.py --config configs/kaggle_coco.yaml \
  --checkpoint /kaggle/working/runs/fcos_r50/best.pth --dump /kaggle/working/results.json
import json; print(json.load(open('/kaggle/working/results.json')))

## Save
Everything in `/kaggle/working` is captured as notebook **Output** on commit — `best.pth`, TensorBoard logs, `results.json`. Download from the Output tab, or **Save Version → Save & Run All (Commit)** to run headless without the idle timeout.

In [ ]:
# 8. (optional) inline TensorBoard
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/runs